# Exploration locale — données contrôlées (jalons 0 à 3)

Ce notebook utilise les mêmes chargeurs que le programme. Il ne produit pas encore
de prédiction de possession. **Exécuter les cellules dans l'ordre** (Maj+Entrée).
Les graphiques/tableaux issus des données privées restent locaux : ne pas publier
de sorties. Effacer toutes les sorties et compteurs avant un commit.

Les cinq minutes publiques servent exclusivement au diagnostic de format,
d'alignement et de sens des labels. Aucun apprentissage, seuil ou choix de modèle
ne doit en dépendre. Voir [le protocole](../docs/protocole.md).

In [ ]:
import sys
from pathlib import Path
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

# Repérer le package depuis la racine ou analysis/, sans imposer un nom de dépôt.
root = next((p for p in [Path.cwd(), *Path.cwd().parents]
             if (p / "src" / "config.py").is_file()), None)
if root is None:
    raise RuntimeError("Ouvrir Jupyter depuis le depot MYA ou analysis/")
if str(root) not in sys.path:
    sys.path.insert(0, str(root))
from src import config
from src.utils import load_atd, load_homography, load_smart_tagging
from src.utils.official import check_official_revision

print("Configuration :", config.MYA_DIR.name)
print("Revision officielle :", check_official_revision())
print("Fichiers disponibles :", all(config.check_data_assets().values()))

## 1. Tracking tactique

Le premier chargement valide toutes les lignes et crée un cache local. Les suivants
vérifient les empreintes des entrées avant de le réutiliser. Les NaN sont conservés.
Les pistes ne sont ni des personnes stables ni des numéros de maillot.

In [ ]:
atd = load_atd("tactical")
print("Joueurs :", atd["players"].shape, "Ballon :", atd["ball"].shape)
print("Cache reutilise :", atd["cache_hit"])
assert np.array_equal(atd["video_frames"], atd["frame_ids"] - 1)
assert set(atd["team_mapping"].values()) == {"H", "A"}
assert np.array_equal(atd["players"][..., 2], atd["players_metres"][..., 2], equal_nan=True)
print("Unite de vitesse XML :", atd["speed_unit"], "; divergence avec DATA.md :", atd["speed_documentation_conflict"])

## 2. Observations absentes et périodes

Une observation manquante n'indique pas automatiquement un ballon mort. Les deux
mi-temps restent séparées ; on ne concatène pas leurs occultations. Les bornes
utilisées ci-dessous sont celles publiées officiellement.

In [ ]:
rows = []
gap_lengths = []
for half, (start, end) in enumerate(config.EVALUATED_INTERVALS, 1):
    missing = ~atd["ball_observed"][start:end]
    edges = np.diff(np.r_[False, missing, False].astype(int))
    gaps = np.flatnonzero(edges == -1) - np.flatnonzero(edges == 1)
    gap_lengths.extend(gaps.tolist())
    rows.append({"periode": half, "images": end - start,
                 "ballon_observe_pct": 100 * (1 - missing.mean()),
                 "occultations": len(gaps)})
display(pd.DataFrame(rows))
sample = np.arange(0, len(atd["ball"]), int(config.FPS))
fig, ax = plt.subplots(figsize=(12, 3))
ax.plot(sample, atd["players_observed"][sample].sum(axis=1), lw=.8)
ax.set(xlabel="Image video tactique", ylabel="Pistes observees",
       title="Observations automatiques, sans correction manuelle")
plt.show()

## 3. Homographie et Smart Tagging

L'homographie va du terrain normalisé vers les pixels. Les événements utilisent
des intervalles `[début, fin[`. N/A et les champs inconnus restent présents ; les
événements sont des indices automatiques, pas la vérité terrain de possession.

In [ ]:
matrices, errors, valid, homography = load_homography("tactical")
smart = load_smart_tagging()
print("Homographie :", matrices.shape, "Evenements :", len(smart))
print("Convention :", smart.attrs["interval_convention"])
display(smart[["code", "start_frame", "end_frame", "team", "team_code"]].head())
# Ce tableau contient des informations privees : effacer les sorties avant publication.

## 4. Vérification de la TV (facultatif)

La TV possède sa propre chronologie et 51 pistes. Le chargement ci-dessous est
facultatif et ne réalise aucune synchronisation avec la vidéo tactique.

In [ ]:
VERIFY_BROADCAST = False
if VERIFY_BROADCAST:
    broadcast = load_atd("broadcast")
    print("TV :", broadcast["players"].shape)
    assert broadcast["players"].shape == (164488, 51, 3)

## 5. Diagnostic public (facultatif, après gel d'une version)

Cette cellule vérifie uniquement le contrat des labels publics. Elle n'entraîne
aucun modèle et ne sélectionne aucune méthode. Les scores ne sont pas calculés
par ce notebook d'ingestion. La chaîne de référence du jalon 2 est disponible séparément.

In [ ]:
RUN_PUBLIC_DIAGNOSTIC = False
if RUN_PUBLIC_DIAGNOSTIC:
    from src.utils import load_ground_truth
    diagnostic = load_ground_truth(public_window_only=True, purpose="diagnostic")
    assert len(diagnostic) == config.PUBLIC_WINDOW_COUNT
    print("Contrat public OK ; usage diagnostic uniquement")

## 6. Caractéristiques du jalon 3

Exécuter d'abord `.venv\Scripts\python.exe -m src.build_features --source all`
depuis la racine du dépôt. Les cellules suivantes relisent l'archive vérifiée,
sans recalculer une autre version des caractéristiques et sans lire de labels.
Un nombre manquant signifie indisponible, jamais une possession ou un arrêt.
Les tableaux et courbes restent privés : effacer les sorties avant publication.
Voir [les conventions](../docs/FEATURES.md).

In [ ]:
from src.build_features import load_matrix

feature_folder = config.CACHE_DIR / "features" / "tactical"
if not (feature_folder / "manifest.json").is_file():
    raise RuntimeError("Construire les caracteristiques avec python -m src.build_features --source all")
features, feature_manifest, feature_schema = load_matrix(feature_folder)
assert features["values"].shape == (config.TOTAL_FRAMES, 116)
print("Images x caracteristiques :", features["values"].shape)
dictionary = pd.DataFrame(feature_schema["features"])
dictionary["images_valides"] = features["valid"].sum(axis=0)
display(dictionary[["name", "unit", "group", "images_valides"]])

In [ ]:
# Exemple local : une caracteristique de mouvement, sans etiquette de possession.
name = "collective_median_speed_mps"
column = features["names"].tolist().index(name)
indices = np.arange(0, config.TOTAL_FRAMES, int(config.FPS))
fig, ax = plt.subplots(figsize=(12, 3))
ax.plot(features["frame_ids"][indices], features["values"][indices, column], lw=.8)
ax.set(xlabel="Image video tactique", ylabel="Vitesse mediane (m/s)",
       title="Activite collective : les interruptions restent manquantes")
plt.show()